In [2]:
%matplotlib widget

import sys
import os
from pathlib import Path

SCRIPT_DIR = Path(os.getcwd()).parent
sys.path.append(os.path.dirname(SCRIPT_DIR))

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from python.fluorophores import FlStatic
from python.fluorophores import FlCollection
from python.psfs import PsfVectorial
from python.estimators import est_qLSQiter2D
from python.estimators import backgroundsubtractor
from python.simulators import Simulator

In [4]:
def Rolling_Average(Data,Window):
    if Window % 2 == 0: Window = Window + 1
    kernel = np.ones(Window)/Window
    
    DataPad = np.zeros(len(Data)+Window-1)
    DataPad[:int(Window/2)] = Data[:int(Window/2)]
    DataPad[int(Window/2):-int(Window/2)] = Data
    DataPad[-int(Window/2):] = Data[-int(Window/2):]

    av = np.correlate(DataPad,kernel,mode='valid')
    return av

## Investigate the effect of background
Causes: imperfect zero, autofluorescence background, nearby fluorophore
XXX background handling of estimator wrong, this makes this not well
interpretable. Use bg calibration and background in estimator

In [5]:
psf_vec = PsfVectorial()
psf_vec.setpinhole(AU=0.8)

pinhole47100, 

c:\Users\joshe\Documents\GitHub\SimuFLUX\python\psfs\library\samples.py:31: RuntimeWarning: invalid value encountered in sqrt
  M = max(50, 7 * sys['NA']**2 / np.sqrt(sys['ns']**2 - sys['NA']**2) * (abs(sys['zo']) + sys['rz']) / sys['lo'])


: 

In [ ]:
sim = Simulator()
laserpower = 10
numberOfLocalizations = 1000
L = 75
probecenter = False
estimatebackground = False
sim.definePattern("donut", psf_vec,
                  phasemask="vortex", 
                  makepattern="orbitscan", 
                  orbitpoints=6, 
                  probecenter=probecenter,
                  orbitL=L,
                  laserpower=laserpower,
                  pointdwelltime=0.1)
sim.defineComponent("estdonut", "estimator", est_qLSQiter2D, parameters=[L, probecenter], dim=(0,1))
sim.defineComponent("bg", "background", backgroundsubtractor, parameters=["background_estimated"])
seq = ["donut", "bg", "estdonut"]
fltestpos = [0, 0, 0]
fl1 = FlStatic()
fl1.pos = fltestpos
sim.fluorophores = fl1


In [ ]:
# no background
sim.background_estimated = 0
fl1.pos = fltestpos
psf_vec.zerooffset = 0
out = sim.runSequence(seq, maxlocs=numberOfLocalizations)
print("no background")
sim.summarize_results(out)

In [ ]:
xs = np.arange(0,70,5)
fig, axs = plt.subplots(1,3,figsize=(9,3))
plt.sca(axs[0])
_ = sim.scan_fov(seq, xs, tag="no bg", ax1="std", linestyle='k')
plt.sca(axs[2])
_ = sim.scan_fov(seq, xs, tag="no bg", ax1="bias", linestyle='k')
plt.sca(axs[1])
_ = sim.scan_fov(seq, xs, tag="no bg", ax1="rmse", linestyle='k')

# imperfect zero

In [ ]:

psf_vec.zerooffset=0.01
# estimate background by scanning centered fluorophore
fl1.pos = [0, 0, 0]
sim.background_estimated = 0
outtest = sim.runSequence(seq, maxlocs=numberOfLocalizations)
bgphot = np.mean(outtest.raw[:,-1])
bgzeroscan = bgphot/outtest.par[0].pointdwelltime
bgestdirect=fl1.brightness*psf_vec.zerooffset*laserpower*psf_vec.PSFs["pinhole63300"].interp([0, 0, 0])   # illustration how zero offset is converted into background
sim.background_estimated=bgzeroscan*estimatebackground
fl1.pos=fltestpos
out=sim.runSequence(seq, maxlocs=numberOfLocalizations)
print("zerooffset: ")
sim.summarize_results(out)

In [ ]:
plt.sca(axs[0])
_ = sim.scan_fov(seq, xs, tag="imperfect zero", ax1="std", linestyle='r--')
plt.sca(axs[2])
_ = sim.scan_fov(seq, xs, tag="imperfect zero", ax1="bias", linestyle='r--')
plt.sca(axs[1])
_ = sim.scan_fov(seq, xs, tag="imperfect zero", ax1="rmse", linestyle='r--')

## Autofluorescence background

In [ ]:
psf_vec.zerooffset = 0.0
sim.background = 30
sim.background_estimated = sim.background*laserpower*estimatebackground  # in general, the GT background is not known but needs to be calibrated 
out = sim.runSequence(seq, maxlocs=numberOfLocalizations)
print("fluorescence background: ")
sim.summarize_results(out)

In [ ]:
plt.sca(axs[0])
_ = sim.scan_fov(seq, xs, tag="auto fluorescence", ax1="std", linestyle='b:')
plt.sca(axs[2])
_ = sim.scan_fov(seq, xs, tag="auto fluorescence", ax1="bias", linestyle='b:')
plt.sca(axs[1])
_ = sim.scan_fov(seq, xs, tag="auto fluorescence", ax1="rmse", linestyle='b:')

## nearby fluorophore

In [ ]:
fl2 = FlStatic()

biases = []
photrates = []
Xs = range(0,500,1)

for i in Xs:
  fl2.pos = [i, 00, 000]

  fc = FlCollection()
  fc.add([fl1, fl2])
  psf_vec.zerooffset = 0
  sim.fluorophores = fc  # make a simulator and attach fluorophore
  sim.background_estimated = 0  # we don't know bg
  out = sim.runSequence(seq, maxlocs=100)#numberOfLocalizations)

  print("nearby fluorophore:",i)
  summ = sim.summarize_results(out)
  print(i,summ.bias[0],summ.photrate)

  biases.append(summ.bias[0])
  photrates.append(summ.photrate)

biases = np.asarray(biases)
photrates = np.asarray(photrates)
plt.close()
plt.plot(Xs,photrates)
plt.show()

In [ ]:
plt.close()
plt.plot(Xs,biases)
plt.show()
print(Rolling_Average(np.asarray(biases),10))


In [ ]:
plt.close()
plt.plot(Xs,np.gradient(biases))